In [1]:
import sys
sys.path.insert(0, '../src')

import pdfplumber
from pathlib import Path
import re
from collections import defaultdict

PDF_DIR = Path('../data/pdfs')
print(f"PDF Directory: {PDF_DIR.resolve()}")

# Récupérer des PDFs de différentes saisons
all_pdfs = list(PDF_DIR.glob('**/*.pdf'))
print(f"Total PDFs: {len(all_pdfs)}")

PDF Directory: /home/vincheetah/Documents/Programmation/Python/PyVolley/data/pdfs
Total PDFs: 116356


In [2]:
# Sélectionner un PDF de test
test_pdf = all_pdfs[0]
print(f"Test PDF: {test_pdf}")

# Ouvrir et analyser la structure
with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    print(f"\nNombre de tables: {len(tables)}")
    for i, table in enumerate(tables):
        print(f"\n{'='*60}")
        print(f"TABLE {i} - {len(table)} lignes")
        print('='*60)
        for j, row in enumerate(table[:30]):  # Afficher les 30 premières lignes
            # Nettoyer les cellules
            cleaned = [str(c)[:20] if c else '' for c in row]
            print(f"  [{j:2d}] {cleaned}")

Test PDF: ../data/pdfs/test/ABCCS_EFA001.pdf

Nombre de tables: 5

TABLE 0 - 44 lignes
  [ 0] ['', '', '', '', '', '', '', '', '', '', '', 'S\nE\nT\n1', '', 'LES NEPTUNES NANTES ', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', 'VOLLEY BALMA QUINT F', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
  [ 1] ['Ordre de Service', '', '', '', '', '', '', '', '', '', '', '', '', 'I', '', 'II', '', '', 'III', '', 'IV', '', 'V', '', 'VI', '', '11121\n21222\n31323\n41', '', '', '', '', 'I', '', 'II', '', 'III', '', 'IV', '', 'V', '', 'VI', '', '1\n2\n3\n4\n5\n6\n7', '', '', '', '', '']
  [ 2] ['Formation de Départ', '', '', '', '', '', '', '', '', '', '', '', '', '4', '', '3', '', '', '8', '', '2', '', '1', '', '11', '', '', '', '', '', '', '10', '', '4', '', '7', '', '11', '', '9', '', '2', '', '', '', '', '', '', '']
  [ 3] ['Remplaçants', '', '', '', '', '', 'Joueur N°', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '

## 1. Exploration des détails de Sets

La table RESULTATS contient généralement:
- Les scores par set
- Les heures de début/fin
- Les positions de départ (I, II, III, IV, V, VI)
- Les changements
- Les timeouts

In [3]:
# Chercher la table RESULTATS (généralement table 4)
def find_resultats_table(tables):
    """Trouve la table des RESULTATS."""
    for i, table in enumerate(tables):
        for row in table:
            if row and any('RESULTATS' in str(c) for c in row if c):
                return i, table
    return None, None

with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    idx, resultats = find_resultats_table(tables)
    if resultats:
        print(f"Table RESULTATS trouvée à l'index {idx}")
        print(f"Nombre de lignes: {len(resultats)}")
        print("\n" + "="*100)
        for j, row in enumerate(resultats):
            print(f"[{j:2d}] {row}")

Table RESULTATS trouvée à l'index 3
Nombre de lignes: 10

[ 0] ['RESULTATS', None, None, None, None, None, None, None, None, None]
[ 1] ['Equipe A', None, None, None, None, 'Equipe B', None, None, None, None]
[ 2] ['T', 'R', 'G', 'P', 'Durée par Set', None, 'P', 'G', 'R', 'T']
[ 3] ['0', '0', '1', '25', "1 20'\n2 31'\n3 26'", None, '7', '0', '0', '2']
[ 4] ['0', '3', '1', '25', None, None, '14', '0', '0', '2']
[ 5] ['0', '5', '1', '25', None, None, '16', '0', '0', '2']
[ 6] ['0', '8', '3', '75', "77'", None, '37', '0', '0', '6']
[ 7] ['Début', None, None, None, 'Fin', None, 'Durée', None, None, None]
[ 8] ['19:58', None, None, None, '21:21', None, '1h22', None, None, None]
[ 9] ['Vainqueur: LES NEPTUNES NANTES VO 3/0', None, None, None, None, None, None, None, None, None]


In [ ]:
# Analyser la structure de la table RESULTATS
def analyze_resultats_structure(resultats):
    """Analyse la structure de la table RESULTATS."""
    structure = {
        'header_row': None,
        'set_rows': [],
        'positions_rows': [],
        'changements_rows': [],
        'timeouts_rows': [],
        'scores_rows': []
    }
    
    for i, row in enumerate(resultats):
        row_str = ' '.join(str(c) for c in row if c)
        
        # Détecter les lignes d'en-tête
        if 'RESULTATS' in row_str:
            structure['header_row'] = i
        
        # Détecter les lignes de position (I, II, III...)
        if any(pos in row_str for pos in ['  I  ', '  II  ', '  III  ', 'Position']):
            structure['positions_rows'].append(i)
        
        # Détecter les lignes avec numéros de set
        if re.search(r'\b(SET|Set)\s*[1-5]\b', row_str):
            structure['set_rows'].append(i)
        
        # Détecter les changements
        if 'Changements' in row_str or 'CHANG' in row_str.upper():
            structure['changements_rows'].append(i)
        
        # Détecter les timeouts
        if 'Temps' in row_str or 'TM' in row_str:
            structure['timeouts_rows'].append(i)
            
        # Détecter les lignes de scores (chiffres séparés)
        if re.search(r'\d{1,2}[/-]\d{1,2}', row_str):
            structure['scores_rows'].append(i)
    
    return structure

if resultats:
    struct = analyze_resultats_structure(resultats)
    print("Structure de la table RESULTATS:")
    for key, value in struct.items():
        print(f"  {key}: {value}")

In [ ]:
# Explorer plusieurs PDFs pour valider la structure
import random

sample_pdfs = random.sample(all_pdfs, min(20, len(all_pdfs)))

structures = []
for pdf_path in sample_pdfs:
    try:
        with pdfplumber.open(pdf_path) as pdf:
            page = pdf.pages[0]
            tables = page.extract_tables()
            idx, resultats = find_resultats_table(tables)
            if resultats:
                structures.append({
                    'path': str(pdf_path),
                    'num_tables': len(tables),
                    'resultats_idx': idx,
                    'resultats_rows': len(resultats),
                    'structure': analyze_resultats_structure(resultats)
                })
    except Exception as e:
        print(f"Erreur {pdf_path.name}: {e}")

print(f"\nAnalysé {len(structures)} PDFs avec table RESULTATS")
for s in structures[:5]:
    print(f"\n{Path(s['path']).name}:")
    print(f"  Tables: {s['num_tables']}, RESULTATS idx: {s['resultats_idx']}, Rows: {s['resultats_rows']}")
    print(f"  Positions: {s['structure']['positions_rows']}")
    print(f"  Sets: {s['structure']['set_rows']}")

## 2. Exploration des Libéros et Officiels

Après la liste des joueurs, il y a généralement:
- Une zone "LIBEROS" avec les numéros des libéros
- Une zone "OFFICIELS" avec l'entraîneur, assistant, etc.

In [4]:
# Analyser la table des équipes (table 3 généralement)
with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    # Table 3 contient les joueurs
    if len(tables) > 3:
        equipes_table = tables[3]
        print(f"Table équipes - {len(equipes_table)} lignes")
        print("="*120)
        
        for j, row in enumerate(equipes_table):
            # Chercher les zones spéciales
            row_str = ' '.join(str(c) for c in row if c)
            marker = ''
            if 'LIBERO' in row_str.upper():
                marker = ' <-- LIBERO'
            elif 'OFFICIEL' in row_str.upper():
                marker = ' <-- OFFICIEL'
            elif 'ENTRAINEUR' in row_str.upper() or 'COACH' in row_str.upper():
                marker = ' <-- COACH'
            
            print(f"[{j:2d}] {row}{marker}")

Table équipes - 10 lignes
[ 0] ['RESULTATS', None, None, None, None, None, None, None, None, None]
[ 1] ['Equipe A', None, None, None, None, 'Equipe B', None, None, None, None]
[ 2] ['T', 'R', 'G', 'P', 'Durée par Set', None, 'P', 'G', 'R', 'T']
[ 3] ['0', '0', '1', '25', "1 20'\n2 31'\n3 26'", None, '7', '0', '0', '2']
[ 4] ['0', '3', '1', '25', None, None, '14', '0', '0', '2']
[ 5] ['0', '5', '1', '25', None, None, '16', '0', '0', '2']
[ 6] ['0', '8', '3', '75', "77'", None, '37', '0', '0', '6']
[ 7] ['Début', None, None, None, 'Fin', None, 'Durée', None, None, None]
[ 8] ['19:58', None, None, None, '21:21', None, '1h22', None, None, None]
[ 9] ['Vainqueur: LES NEPTUNES NANTES VO 3/0', None, None, None, None, None, None, None, None, None]


In [5]:
# Exploration détaillée de TOUTES les tables
with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    print(f"Nombre total de tables: {len(tables)}")
    for t_idx, table in enumerate(tables):
        print(f"\n{'='*80}")
        print(f"TABLE {t_idx} - {len(table)} lignes x {len(table[0]) if table else 0} colonnes")
        print('='*80)
        
        for r_idx, row in enumerate(table[:20]):  # Limiter à 20 lignes
            # Marquer les lignes importantes
            row_str = ' '.join(str(c) for c in row if c).upper()
            marker = ''
            if 'LIBERO' in row_str:
                marker = ' <-- LIBERO'
            elif 'OFFICIEL' in row_str:
                marker = ' <-- OFFICIEL'
            elif 'CAPITAINE' in row_str:
                marker = ' <-- CAPITAINE'
            elif 'ENTRAINEUR' in row_str or 'COACH' in row_str:
                marker = ' <-- COACH'
            elif 'SANCTION' in row_str:
                marker = ' <-- SANCTION'
            elif 'AVERTISSEMENT' in row_str or 'PENALITE' in row_str:
                marker = ' <-- SANCTION_DATA'
            
            # Nettoyer et afficher
            cleaned = [str(c)[:15] if c else '' for c in row]
            print(f"[{r_idx:2d}] {cleaned}{marker}")

Nombre total de tables: 5

TABLE 0 - 44 lignes x 49 colonnes
[ 0] ['', '', '', '', '', '', '', '', '', '', '', 'S\nE\nT\n1', '', 'LES NEPTUNES NA', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', 'VOLLEY BALMA QU', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
[ 1] ['Ordre de Servic', '', '', '', '', '', '', '', '', '', '', '', '', 'I', '', 'II', '', '', 'III', '', 'IV', '', 'V', '', 'VI', '', '11121\n21222\n313', '', '', '', '', 'I', '', 'II', '', 'III', '', 'IV', '', 'V', '', 'VI', '', '1\n2\n3\n4\n5\n6\n7', '', '', '', '', '']
[ 2] ['Formation de Dé', '', '', '', '', '', '', '', '', '', '', '', '', '4', '', '3', '', '', '8', '', '2', '', '1', '', '11', '', '', '', '', '', '', '10', '', '4', '', '7', '', '11', '', '9', '', '2', '', '', '', '', '', '', '']
[ 3] ['Remplaçants', '', '', '', '', '', 'Joueur N°', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', 

In [6]:
# Chercher spécifiquement les lignes LIBERO et OFFICIEL dans toutes les tables
with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    print(f"Recherche des lignes clés dans {len(tables)} tables\n")
    
    for t_idx, table in enumerate(tables):
        found_lines = []
        for r_idx, row in enumerate(table):
            row_str = ' '.join(str(c) for c in row if c).upper()
            
            # Chercher les mots-clés importants
            keywords = ['LIBERO', 'OFFICIEL', 'ENTRAINEUR', 'ASSISTANT', 'CAPITAINE', 
                       'SANCTION', 'AVERTISSEMENT', 'PENALITE', 'REMARQUE']
            
            for kw in keywords:
                if kw in row_str:
                    found_lines.append((r_idx, kw, row))
                    break
        
        if found_lines:
            print(f"TABLE {t_idx}:")
            for r_idx, kw, row in found_lines:
                print(f"  [{r_idx:2d}] {kw}: {row}")

Recherche des lignes clés dans 5 tables

TABLE 0:
  [30] SANCTION: ['SANCTIONS', None, None, None, None, None, 'DEMANDE NON FONDEE', None, None, None, None, None, None, 'REMARQUES', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
  [43] CAPITAINE: [None, None, None, None, None, None, None, None, None, None, None, None, None, '', None, None, None, None, None, None, None, None, None, None, None, None, 'Capitaines', None, None, None, '', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
TABLE 2:
  [ 3] LIBERO: ['LIBEROS', None, None, None, None, None]
TABLE 4:
  [ 1] CAPITAINE: ['Capitaine', 'Capitaine']
  [ 2] ENTRAINEUR: ['Entraineur', 'Entraineur']


In [7]:
# Explorer TABLE 2 (LIBEROS) en détail
with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    if len(tables) > 2:
        table2 = tables[2]
        print(f"TABLE 2 - {len(table2)} lignes")
        print("="*100)
        for i, row in enumerate(table2):
            print(f"[{i:2d}] {row}")

TABLE 2 - 4 lignes
[ 0] ['LES NEPTUNES NANTES VOLLEY ASS', None, None, 'VOLLEY BALMA QUINT FONSEGRIVES', None, None]
[ 1] ['N°', 'Nom Prénom', 'Licence', 'N°', 'Nom Prénom', 'Licence']
[ 2] ['01 KADIKU BRIANNA 2499240\n02 DAVAI ANA 2148224\n03 LANDAIS SALOME 2251841\n04 GLYNN SARAH 2876307\n05 MPOUPE ELISA 2362147\n06 ANSQUER YUNA 2252009\n07 GOMBEAU NINON 2239971\n08 JUND MARIE 2304027\n10 VIDALLER STELLA 1866985\n11 CHAMEAUX LENA 1778916\n18 MOY ANAELLE 2302857', None, None, '02 FONTAINE MARINA 1830746\n04 HERLIN OCEANE 2099511\n07 SENTENAC BAHIA 2112025\n09 LAURENT MANON 1797686\n10 QUINZONI JULIE 1821312\n11 ALBOUY NINE 2458001\n23 VALEFAKAAGA SYARHA 2689927\n81 POUMOT INES 2089822', None, None]
[ 3] ['LIBEROS', None, None, None, None, None]


In [8]:
# Explorer TABLE 4 (Officiels) en détail
with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    if len(tables) > 4:
        table4 = tables[4]
        print(f"TABLE 4 - {len(table4)} lignes")
        print("="*100)
        for i, row in enumerate(table4):
            print(f"[{i:2d}] {row}")

TABLE 4 - 3 lignes
[ 0] ['SIGNATURES', None]
[ 1] ['Capitaine', 'Capitaine']
[ 2] ['Entraineur', 'Entraineur']


In [9]:
# Explorer TABLE 0 (principale avec sanctions) - autour des lignes intéressantes
with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    if tables:
        table0 = tables[0]
        print(f"TABLE 0 - {len(table0)} lignes")
        
        # Afficher autour de la ligne SANCTIONS (row 30)
        print("\n=== Zone SANCTIONS (rows 28-35) ===")
        for i in range(28, min(36, len(table0))):
            row = table0[i]
            # Filtrer les None pour lisibilité
            filtered = [str(c)[:20] if c else '' for c in row]
            non_empty = [(j, c) for j, c in enumerate(filtered) if c]
            print(f"[{i:2d}] {non_empty}")

TABLE 0 - 44 lignes

=== Zone SANCTIONS (rows 28-35) ===
[28] []
[29] []
[30] [(0, 'SANCTIONS'), (6, 'DEMANDE NON FONDEE'), (13, 'REMARQUES')]
[31] [(6, 'EQU.A EQU.B')]
[32] [(0, 'A'), (1, 'P'), (3, 'E'), (5, 'D'), (6, 'A/B'), (8, 'Set'), (9, 'Score')]
[33] []
[34] [(13, 'APPROBATION')]
[35] [(13, 'Arbitres'), (16, 'NOM Prénom'), (26, 'Ligue'), (30, 'Licence'), (33, 'Signature')]


In [11]:
# Analyser un PDF avec sanctions
sanction_pdf = pdfs_with_sanctions[0] if pdfs_with_sanctions else None
if sanction_pdf:
    print(f"Analyse de: {sanction_pdf}")
    
    with pdfplumber.open(sanction_pdf) as pdf:
        page = pdf.pages[0]
        tables = page.extract_tables()
        
        if tables:
            table0 = tables[0]
            print(f"\n=== TABLE 0 - Zone SANCTIONS ===")
            
            # Trouver la ligne SANCTIONS
            sanction_start = None
            for i, row in enumerate(table0):
                row_str = ' '.join(str(c) for c in row if c).upper()
                if 'SANCTION' in row_str:
                    sanction_start = i
                    break
            
            if sanction_start:
                print(f"Zone sanctions commence à la ligne {sanction_start}")
                for i in range(max(0, sanction_start-2), min(sanction_start+10, len(table0))):
                    row = table0[i]
                    filtered = [(j, str(c)[:25]) for j, c in enumerate(row) if c]
                    print(f"[{i:2d}] {filtered}")

Analyse de: ../data/pdfs/test/ABCCS_EFA001.pdf

=== TABLE 0 - Zone SANCTIONS ===
Zone sanctions commence à la ligne 30
[28] []
[29] []
[30] [(0, 'SANCTIONS'), (6, 'DEMANDE NON FONDEE'), (13, 'REMARQUES')]
[31] [(6, 'EQU.A EQU.B')]
[32] [(0, 'A'), (1, 'P'), (3, 'E'), (5, 'D'), (6, 'A/B'), (8, 'Set'), (9, 'Score')]
[33] []
[34] [(13, 'APPROBATION')]
[35] [(13, 'Arbitres'), (16, 'NOM Prénom'), (26, 'Ligue'), (30, 'Licence'), (33, 'Signature')]
[36] [(13, '1er'), (16, "GHENIMI NOUR'AMIR"), (26, 'NAQ'), (30, '1420177')]
[37] [(13, '2ème'), (16, 'ABBAS MEHDI'), (26, 'NOR'), (30, '1875296')]
[38] [(13, 'Marqueur'), (16, 'ANDRE ANAÏS'), (26, 'PDL'), (30, '1943464')]
[39] [(13, 'Marq.Ass.')]


In [12]:
# Chercher des PDFs avec sanctions non-vides (vérifier si la ligne 33 contient des données)
def find_pdf_with_real_sanctions(pdf_list, max_check=500):
    """Cherche des PDFs avec de vraies sanctions remplies."""
    found = []
    for pdf_path in pdf_list[:max_check]:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                page = pdf.pages[0]
                tables = page.extract_tables()
                
                if tables:
                    table0 = tables[0]
                    # Chercher la ligne après les headers de sanctions (généralement row 33)
                    for i, row in enumerate(table0):
                        row_str = ' '.join(str(c) for c in row if c).upper()
                        # Chercher A/P/E/D suivis de données
                        if i >= 32 and i <= 35:
                            # Vérifier s'il y a des données de sanctions (numéro de joueur, set, score)
                            cells = [str(c) for c in row if c]
                            for cell in cells:
                                # Chercher pattern de sanction: numéro joueur, set, score
                                if re.search(r'\d+[A-B]?\s+\d+\s+\d+-\d+', cell):
                                    found.append(pdf_path)
                                    break
                            if pdf_path in found:
                                break
        except:
            pass
        
        if len(found) >= 5:
            break
    
    return found

real_sanctions = find_pdf_with_real_sanctions(all_pdfs)
print(f"PDFs avec sanctions réelles trouvées: {len(real_sanctions)}")
for p in real_sanctions:
    print(f"  - {p}")

PDFs avec sanctions réelles trouvées: 0


In [13]:
# Méthode alternative: chercher dans le texte brut
import random

def check_sanctions_in_text(pdf_path):
    """Vérifie si un PDF contient des sanctions dans le texte."""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = pdf.pages[0].extract_text() or ''
            
            # Patterns de sanctions typiques: "04 A 1 12-8" (joueur 4, Avertissement, set 1, score 12-8)
            patterns = [
                r'\b\d{1,2}\s+[APED]\s+\d\s+\d{1,2}-\d{1,2}',  # "04 A 1 12-8"
                r'Avertissement\s+\d+',  # "Avertissement 4"
                r'Pénalité\s+\d+',  # "Pénalité 7"
            ]
            
            for pattern in patterns:
                if re.search(pattern, text, re.IGNORECASE):
                    return True, pattern
            
            # Chercher dans les colonnes A/P/E/D si elles ont des numéros
            lines = text.split('\n')
            for line in lines:
                # Ligne avec données de sanction
                if re.match(r'^\d{1,2}\s+[A-B]?\s*\d\s+\d{1,2}[-/]\d{1,2}', line.strip()):
                    return True, line.strip()
                    
    except Exception as e:
        return False, str(e)
    
    return False, None

# Test sur un échantillon
sample = random.sample(all_pdfs, min(200, len(all_pdfs)))
found_sanctions = []

for pdf in sample:
    has_sanction, info = check_sanctions_in_text(pdf)
    if has_sanction:
        found_sanctions.append((pdf, info))

print(f"PDFs avec sanctions (sur {len(sample)} testés): {len(found_sanctions)}")
for p, info in found_sanctions[:5]:
    print(f"  - {p.name}: {info[:50] if info else 'N/A'}...")

PDFs avec sanctions (sur 200 testés): 0


In [14]:
# Analyser en profondeur la structure des RESULTATS (sets)
# La Table 3 (ou RESULTATS) contient les infos de sets

with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    # Trouver la table RESULTATS
    for t_idx, table in enumerate(tables):
        for row in table:
            if any('RESULTATS' in str(c) for c in row if c):
                print(f"=== TABLE {t_idx} - RESULTATS ===")
                print(f"Nombre de lignes: {len(table)}")
                print()
                
                for r_idx, row in enumerate(table):
                    print(f"[{r_idx:2d}] {row}")
                break

=== TABLE 3 - RESULTATS ===
Nombre de lignes: 10

[ 0] ['RESULTATS', None, None, None, None, None, None, None, None, None]
[ 1] ['Equipe A', None, None, None, None, 'Equipe B', None, None, None, None]
[ 2] ['T', 'R', 'G', 'P', 'Durée par Set', None, 'P', 'G', 'R', 'T']
[ 3] ['0', '0', '1', '25', "1 20'\n2 31'\n3 26'", None, '7', '0', '0', '2']
[ 4] ['0', '3', '1', '25', None, None, '14', '0', '0', '2']
[ 5] ['0', '5', '1', '25', None, None, '16', '0', '0', '2']
[ 6] ['0', '8', '3', '75', "77'", None, '37', '0', '0', '6']
[ 7] ['Début', None, None, None, 'Fin', None, 'Durée', None, None, None]
[ 8] ['19:58', None, None, None, '21:21', None, '1h22', None, None, None]
[ 9] ['Vainqueur: LES NEPTUNES NANTES VO 3/0', None, None, None, None, None, None, None, None, None]


In [15]:
# Chercher les positions de départ et changements dans le PDF
# Ces infos sont probablement dans une autre partie du document

with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    
    # Extraire tout le texte avec layout
    text = page.extract_text(layout=True) or ''
    
    # Chercher les lignes avec Position, Changement, Temps mort
    print("=== Recherche de mots-clés dans le texte ===")
    keywords = ['Position', 'Changement', 'Temps', 'Service', 'Formation', 
                'I II III', 'IV V VI', 'Remplacement']
    
    lines = text.split('\n')
    for i, line in enumerate(lines):
        for kw in keywords:
            if kw.lower() in line.lower():
                print(f"[{i:3d}] {line[:100]}")

=== Recherche de mots-clés dans le texte ===
[  7]          Ordre de Service E I II III IV V VI 1 2 1 1 1 2 2 2 1 2 I II III IV V VI 1 2 E I II III IV 
[  7]          Ordre de Service E I II III IV V VI 1 2 1 1 1 2 2 2 1 2 I II III IV V VI 1 2 E I II III IV 
[  7]          Ordre de Service E I II III IV V VI 1 2 1 1 1 2 2 2 1 2 I II III IV V VI 1 2 E I II III IV 
[  8]        Formation de Départ 4 3 8 2 1 11 31323 10 4 7 11 9 2 3    2 10 4  7  11 9 313  3  8  2  1 11 
[ 15]   Tours au service                                                                                  
[ 19]          Ordre de Service E I II III IV V VI 1 2 1 1 1 2 2 2 1 2 I II III IV V VI 1 2 1 1 1 2 E I II
[ 19]          Ordre de Service E I II III IV V VI 1 2 1 1 1 2 2 2 1 2 I II III IV V VI 1 2 1 1 1 2 E I II
[ 19]          Ordre de Service E I II III IV V VI 1 2 1 1 1 2 2 2 1 2 I II III IV V VI 1 2 1 1 1 2 E I II
[ 20]        Formation de Départ 11 4 3 8 2 1 31323 10 4 23 11 9 2 313                             

In [16]:
# Analyser Table 0 pour les zones de sets (formations, changements)
with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    if tables:
        table0 = tables[0]
        
        # Chercher les zones de sets
        print(f"TABLE 0 - {len(table0)} lignes")
        print("Recherche des zones de sets...")
        
        for i, row in enumerate(table0):
            row_str = ' '.join(str(c) for c in row if c).upper()
            
            # Chercher les indicateurs de sets
            if any(kw in row_str for kw in ['SET', 'FORMATION', 'SERVICE', 'TOURS', 'POSITION']):
                filtered = [(j, str(c)[:20]) for j, c in enumerate(row) if c]
                print(f"[{i:2d}] {filtered}")

TABLE 0 - 44 lignes
Recherche des zones de sets...
[ 1] [(0, 'Ordre de Service'), (13, 'I'), (15, 'II'), (18, 'III'), (20, 'IV'), (22, 'V'), (24, 'VI'), (26, '11121\n21222\n31323\n41'), (31, 'I'), (33, 'II'), (35, 'III'), (37, 'IV'), (39, 'V'), (41, 'VI'), (43, '1\n2\n3\n4\n5\n6\n7')]
[ 2] [(0, 'Formation de Départ'), (13, '4'), (15, '3'), (18, '8'), (20, '2'), (22, '1'), (24, '11'), (31, '10'), (33, '4'), (35, '7'), (37, '11'), (39, '9'), (41, '2')]
[ 6] [(0, 'Tours au service'), (6, '1'), (8, '5'), (13, '2'), (15, '8'), (18, '9'), (20, '11'), (22, '15'), (24, '18'), (31, 'X'), (33, '1'), (35, '2'), (37, '4'), (39, '5'), (41, '6')]
[11] [(0, 'Ordre de Service'), (13, 'I'), (15, 'II'), (18, 'III'), (20, 'IV'), (22, 'V'), (24, 'VI'), (26, '11121\n21222\n31323\n41'), (31, 'I'), (33, 'II'), (35, 'III'), (37, 'IV'), (39, 'V'), (41, 'VI'), (43, '111\n212\n313\n414\n515\n')]
[12] [(0, 'Formation de Départ'), (13, '11'), (15, '4'), (18, '3'), (20, '8'), (22, '2'), (24, '1'), (31, '10'), (33, 

In [17]:
# Analyser les lignes entre les formations et les tours au service (changements)
with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    if tables:
        table0 = tables[0]
        
        print("=== Lignes de SET 1 (rows 0-10) ===")
        for i in range(0, 11):
            row = table0[i]
            filtered = [(j, str(c)[:15]) for j, c in enumerate(row) if c]
            if filtered:
                print(f"[{i:2d}] {filtered}")
        
        print("\n=== Lignes de SET 2 (rows 11-20) ===")
        for i in range(11, min(21, len(table0))):
            row = table0[i]
            filtered = [(j, str(c)[:15]) for j, c in enumerate(row) if c]
            if filtered:
                print(f"[{i:2d}] {filtered}")

=== Lignes de SET 1 (rows 0-10) ===
[ 0] [(11, 'S\nE\nT\n1'), (13, 'LES NEPTUNES NA'), (31, 'VOLLEY BALMA QU')]
[ 1] [(0, 'Ordre de Servic'), (13, 'I'), (15, 'II'), (18, 'III'), (20, 'IV'), (22, 'V'), (24, 'VI'), (26, '11121\n21222\n313'), (31, 'I'), (33, 'II'), (35, 'III'), (37, 'IV'), (39, 'V'), (41, 'VI'), (43, '1\n2\n3\n4\n5\n6\n7')]
[ 2] [(0, 'Formation de Dé'), (13, '4'), (15, '3'), (18, '8'), (20, '2'), (22, '1'), (24, '11'), (31, '10'), (33, '4'), (35, '7'), (37, '11'), (39, '9'), (41, '2')]
[ 3] [(0, 'Remplaçants'), (6, 'Joueur N°')]
[ 4] [(6, 'Score')]
[ 6] [(0, 'Tours au servic'), (6, '1'), (8, '5'), (13, '2'), (15, '8'), (18, '9'), (20, '11'), (22, '15'), (24, '18'), (31, 'X'), (33, '1'), (35, '2'), (37, '4'), (39, '5'), (41, '6')]
[ 7] [(6, '2'), (8, '6'), (13, '25'), (26, 'T'), (31, '7'), (43, 'T')]
[ 8] [(6, '3'), (8, '7'), (43, '1:6')]
[ 9] [(6, '4'), (8, '8'), (43, '5:15')]
[10] [(11, 'S\nE\nT\n3'), (13, 'LES NEPTUNES NA'), (31, 'VOLLEY BALMA QU')]

=== Lignes de SET 2

In [ ]:
# Fonction pour extraire les libéros
def extract_liberos_from_table(table):
    """Extrait les numéros des libéros de la table."""
    liberos_a = []
    liberos_b = []
    
    for row in table:
        row_str = ' '.join(str(c) for c in row if c).upper()
        
        # Chercher les lignes avec LIBERO
        if 'LIBERO' in row_str:
            # Les numéros de libéros sont souvent dans les colonnes adjacentes
            for i, cell in enumerate(row):
                if cell and re.match(r'^\d{1,2}$', str(cell).strip()):
                    # Déterminer si c'est équipe A ou B basé sur la position
                    if i < len(row) // 2:
                        liberos_a.append(str(cell).strip())
                    else:
                        liberos_b.append(str(cell).strip())
    
    return liberos_a, liberos_b

# Fonction pour extraire les officiels
def extract_officiels_from_table(table):
    """Extrait les officiels (entraîneur, etc.)."""
    officiels_a = []
    officiels_b = []
    
    capture = False
    for row in table:
        row_str = ' '.join(str(c) for c in row if c).upper()
        
        # Détecter le début de la zone officiels
        if 'OFFICIEL' in row_str or 'ENTRAINEUR' in row_str:
            capture = True
        
        if capture:
            # Chercher les rôles et noms
            for i, cell in enumerate(row):
                if cell:
                    cell_str = str(cell).strip()
                    if any(role in cell_str.upper() for role in ['ENTRAINEUR', 'ASSISTANT', 'MANAGER', 'KINE', 'MEDECIN']):
                        # Le nom suit généralement
                        if i + 1 < len(row) and row[i + 1]:
                            official = {'role': cell_str, 'nom': str(row[i + 1]).strip()}
                            if i < len(row) // 2:
                                officiels_a.append(official)
                            else:
                                officiels_b.append(official)
    
    return officiels_a, officiels_b

# Tester
if len(tables) > 3:
    lib_a, lib_b = extract_liberos_from_table(tables[3])
    print(f"Libéros équipe A: {lib_a}")
    print(f"Libéros équipe B: {lib_b}")
    
    off_a, off_b = extract_officiels_from_table(tables[3])
    print(f"\nOfficiels équipe A: {off_a}")
    print(f"Officiels équipe B: {off_b}")

## 3. Exploration des Sanctions

Les sanctions sont généralement dans une zone séparée du PDF avec:
- Type de sanction (Avertissement, Pénalité, Expulsion, etc.)
- Numéro du joueur
- Set et score au moment de la sanction

In [10]:
# Chercher des PDFs avec des sanctions
def has_sanctions(pdf_path):
    """Vérifie si un PDF contient des sanctions."""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = pdf.pages[0].extract_text() or ''
            # Chercher des indices de sanctions
            sanction_keywords = ['AVERTISSEMENT', 'PENALITE', 'PÉNALITÉ', 'EXPULSION', 
                                'DISQUALIFICATION', 'SANCTION', 'CARTON']
            return any(kw in text.upper() for kw in sanction_keywords)
    except:
        return False

# Rechercher des PDFs avec sanctions
pdfs_with_sanctions = []
for pdf in all_pdfs[:500]:  # Limiter pour performance
    if has_sanctions(pdf):
        pdfs_with_sanctions.append(pdf)
        if len(pdfs_with_sanctions) >= 10:
            break

print(f"PDFs avec potentielles sanctions: {len(pdfs_with_sanctions)}")
for p in pdfs_with_sanctions:
    print(f"  - {p.name}")

PDFs avec potentielles sanctions: 10
  - ABCCS_EFA001.pdf
  - PTIDF92_LME056.pdf
  - PTIDF92_LME017.pdf
  - PTIDF92_LME028.pdf
  - PTIDF92_LME047.pdf
  - PTIDF92_LME044.pdf
  - PTIDF92_LME060.pdf
  - PTIDF92_LME074.pdf
  - PTIDF92_LME005.pdf
  - PTIDF92_LME087.pdf


In [ ]:
# Analyser un PDF avec sanctions
if pdfs_with_sanctions:
    sanction_pdf = pdfs_with_sanctions[0]
    print(f"Analyse de: {sanction_pdf}")
    
    with pdfplumber.open(sanction_pdf) as pdf:
        page = pdf.pages[0]
        
        # Extraire le texte complet
        full_text = page.extract_text() or ''
        print("\n=== Texte contenant 'sanction' ou similaire ===")
        for line in full_text.split('\n'):
            if any(kw in line.upper() for kw in ['AVERT', 'PENAL', 'SANCTION', 'EXPUL', 'CARTON']):
                print(f"  {line}")
        
        # Analyser les tables pour les sanctions
        tables = page.extract_tables()
        print(f"\n=== Tables ===")
        for i, table in enumerate(tables):
            for j, row in enumerate(table):
                row_str = ' '.join(str(c) for c in row if c)
                if any(kw in row_str.upper() for kw in ['AVERT', 'PENAL', 'SANCTION', 'EXPUL', 'CARTON']):
                    print(f"  Table {i}, Row {j}: {row}")

In [ ]:
# Explorer la zone des sanctions dans la table 0
if pdfs_with_sanctions:
    with pdfplumber.open(sanction_pdf) as pdf:
        tables = pdf.pages[0].extract_tables()
        
        # Table 0 contient souvent les sanctions
        if tables:
            table0 = tables[0]
            print(f"Table 0 - Recherche zone sanctions")
            print("="*120)
            
            in_sanctions = False
            for j, row in enumerate(table0):
                row_str = ' '.join(str(c) for c in row if c)
                
                # Détecter la zone sanctions
                if 'SANCTION' in row_str.upper():
                    in_sanctions = True
                
                if in_sanctions or any(kw in row_str.upper() for kw in ['AVERT', 'PENAL', 'W', 'P', 'E']):
                    print(f"[{j:2d}] {row}")

## 4. Exploration des positions de départ et changements

In [ ]:
# Analyser en détail la structure des sets
with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    # Chercher la table des résultats par set
    idx, resultats = find_resultats_table(tables)
    if resultats:
        print("Analyse détaillée de la table RESULTATS:")
        print("="*120)
        
        # Identifier les colonnes
        for j, row in enumerate(resultats):
            # Afficher avec indices de colonnes
            print(f"\n[Row {j:2d}]")
            for i, cell in enumerate(row):
                if cell:
                    print(f"  Col {i:2d}: '{cell}'")

In [ ]:
# Fonction pour parser les détails d'un set
def parse_set_details(resultats_table, set_number):
    """Parse les détails d'un set spécifique."""
    details = {
        'numero': set_number,
        'score_a': None,
        'score_b': None,
        'heure_debut': None,
        'heure_fin': None,
        'duree': None,
        'position_a': [],  # Positions I à VI
        'position_b': [],
        'changements_a': [],
        'changements_b': [],
        'timeouts_a': [],  # Liste de (score_a, score_b)
        'timeouts_b': [],
        'service_initial': None  # 'A' ou 'B'
    }
    
    # Parcourir la table pour trouver les infos du set
    for row in resultats_table:
        row_str = ' '.join(str(c) for c in row if c)
        
        # Chercher le numéro de set
        if f'Set {set_number}' in row_str or f'SET {set_number}' in row_str:
            # Extraire les scores finaux
            score_match = re.search(r'(\d{1,2})\s*[/-]\s*(\d{1,2})', row_str)
            if score_match:
                details['score_a'] = int(score_match.group(1))
                details['score_b'] = int(score_match.group(2))
            
            # Extraire les heures
            time_matches = re.findall(r'(\d{1,2}[h:]\d{2})', row_str)
            if len(time_matches) >= 2:
                details['heure_debut'] = time_matches[0]
                details['heure_fin'] = time_matches[1]
    
    return details

# Tester
if resultats:
    for set_num in range(1, 6):
        details = parse_set_details(resultats, set_num)
        if details['score_a'] is not None:
            print(f"Set {set_num}: {details['score_a']}-{details['score_b']} ({details['heure_debut']} - {details['heure_fin']})")

In [ ]:
# Explorer les zones de positions et changements avec les coordonnées
with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    
    # Utiliser extract_text avec layout pour voir la disposition
    text = page.extract_text(layout=True)
    
    print("=== Texte avec layout (extrait sur RESULTATS) ===")
    lines = text.split('\n')
    in_resultats = False
    for i, line in enumerate(lines):
        if 'RESULTATS' in line:
            in_resultats = True
        if in_resultats:
            print(f"[{i:3d}] {line}")
            if 'REMARQUES' in line or i > 150:  # Limiter
                break

In [ ]:
# Analyser les rectangles et zones du PDF
with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    
    # Récupérer tous les mots avec leurs coordonnées
    words = page.extract_words()
    
    # Chercher les mots liés aux sets
    set_words = [w for w in words if any(kw in w['text'].upper() for kw in ['SET', 'POSITION', 'CHANGEMENT', 'TEMPS'])]
    
    print(f"Mots liés aux sets trouvés: {len(set_words)}")
    for w in set_words:
        print(f"  '{w['text']}' at ({w['x0']:.0f}, {w['top']:.0f})")

In [ ]:
# Définir les zones approximatives pour chaque set basé sur les coordonnées
# Analyser la structure y du document

with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    words = page.extract_words()
    
    # Trouver les lignes horizontales de texte
    y_positions = defaultdict(list)
    for w in words:
        y = round(w['top'] / 5) * 5  # Grouper par 5 pixels
        y_positions[y].append(w['text'])
    
    # Afficher les lignes avec du contenu intéressant
    print("Lignes par position Y:")
    for y in sorted(y_positions.keys()):
        line_text = ' '.join(y_positions[y])
        if any(kw in line_text.upper() for kw in ['SET', 'SCORE', 'POSITION', '25', '24', '23']):
            print(f"Y={y:4.0f}: {line_text[:100]}")

## 5. Analyse complète d'un PDF pour validation

In [ ]:
# Créer une analyse complète d'un PDF
def full_pdf_analysis(pdf_path):
    """Analyse complète d'un PDF pour extraction des données."""
    analysis = {
        'path': str(pdf_path),
        'tables_count': 0,
        'joueurs_a': [],
        'joueurs_b': [],
        'liberos_a': [],
        'liberos_b': [],
        'officiels_a': [],
        'officiels_b': [],
        'sets': [],
        'sanctions': [],
        'capitaine_a': None,
        'capitaine_b': None
    }
    
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[0]
        tables = page.extract_tables()
        analysis['tables_count'] = len(tables)
        
        # Analyser la table des joueurs (table 3)
        if len(tables) > 3:
            joueurs_table = tables[3]
            
            # Chercher les zones libéros
            for row in joueurs_table:
                row_str = ' '.join(str(c) for c in row if c).upper()
                
                if 'LIBERO' in row_str:
                    # Extraire les numéros
                    for i, cell in enumerate(row):
                        if cell and re.match(r'^\d{1,2}$', str(cell).strip()):
                            if i < 10:  # Approximation équipe A
                                analysis['liberos_a'].append(str(cell).strip())
                            else:
                                analysis['liberos_b'].append(str(cell).strip())
        
        # Analyser la table des résultats (table 4 ou similaire)
        idx, resultats = find_resultats_table(tables)
        if resultats:
            # Parser les sets
            for row in resultats:
                row_str = ' '.join(str(c) for c in row if c)
                
                # Chercher les lignes de score de set
                score_match = re.search(r'(\d{1,2})\s*[-/]\s*(\d{1,2})', row_str)
                if score_match and (int(score_match.group(1)) >= 20 or int(score_match.group(2)) >= 20):
                    analysis['sets'].append({
                        'score_a': int(score_match.group(1)),
                        'score_b': int(score_match.group(2))
                    })
        
        # Analyser les sanctions (table 0)
        if tables:
            for row in tables[0]:
                row_str = ' '.join(str(c) for c in row if c).upper()
                if any(kw in row_str for kw in ['AVERTISSEMENT', 'PENALITE', 'PÉNALITÉ', 'EXPULSION']):
                    analysis['sanctions'].append(row)
    
    return analysis

# Tester sur plusieurs PDFs
test_pdfs = random.sample(all_pdfs, min(10, len(all_pdfs)))
for pdf in test_pdfs[:3]:
    print(f"\n{'='*60}")
    print(f"PDF: {pdf.name}")
    result = full_pdf_analysis(pdf)
    print(f"  Tables: {result['tables_count']}")
    print(f"  Libéros A: {result['liberos_a']}, B: {result['liberos_b']}")
    print(f"  Sets trouvés: {len(result['sets'])}")
    for i, s in enumerate(result['sets'], 1):
        print(f"    Set {i}: {s['score_a']}-{s['score_b']}")
    if result['sanctions']:
        print(f"  Sanctions: {len(result['sanctions'])}")

In [ ]:
# Explorer la structure des numéros entourés (capitaines)
# Dans les PDFs, les numéros entourés apparaissent souvent entre parenthèses ou avec caractères spéciaux

with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    
    # Récupérer les caractères
    chars = page.chars
    
    # Chercher des caractères spéciaux autour des numéros
    print(f"Total caractères: {len(chars)}")
    
    # Chercher les parenthèses ou cercles
    special_chars = [c for c in chars if c['text'] in '()○●◯⃝']
    print(f"Caractères spéciaux trouvés: {len(special_chars)}")
    
    # Analyser les fontes des numéros
    number_chars = [c for c in chars if c['text'].isdigit()]
    fonts = set(c.get('fontname', '') for c in number_chars)
    print(f"\nFontes utilisées pour les chiffres: {fonts}")

In [ ]:
# Analyser les graphiques/dessins dans le PDF (cercles autour des numéros)
with pdfplumber.open(test_pdf) as pdf:
    page = pdf.pages[0]
    
    # Récupérer les courbes (curves) qui pourraient être des cercles
    curves = page.curves if hasattr(page, 'curves') else []
    rects = page.rects if hasattr(page, 'rects') else []
    lines = page.lines if hasattr(page, 'lines') else []
    
    print(f"Curves: {len(curves)}")
    print(f"Rects: {len(rects)}")
    print(f"Lines: {len(lines)}")
    
    # Les cercles autour des numéros de capitaines sont souvent des ellipses
    # Chercher dans les objets graphiques
    if curves:
        print("\nExemples de curves:")
        for c in curves[:5]:
            print(f"  {c}")